# Glassdoor Review Analysis
**Text Cleaning  →  Sentiment Scoring (TextBlob + VADER)  →  Model Comparison  →  Validation  →  Statistical Tests  →  Keyword Analysis  →  Visualizations  →  Export**

---

**Dataset:** Amazon Glassdoor reviews  
**Text columns analysed:** `advice_to_management`, `review_pros`, `review_cons`  
**Validation anchor:** `rating_overall` (1–5 stars)  
**Sentiment models:** TextBlob (lexicon) + VADER (lexicon, rule-based)  

**Author:** Henry Oyewumi  
**Date:** March 2025

## Step 1 — Install & Import

In [ ]:
!pip install textblob wordcloud nltk vaderSentiment -q
!python -m textblob.download_corpora -q

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from collections import Counter
from scipy import stats
from scipy.stats import ttest_ind, kruskal
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from wordcloud import WordCloud
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)

vader = SentimentIntensityAnalyzer()
print('Ready.')

## Step 2 — Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'{df.shape[0]} reviews loaded, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print()
text_cols  = ['advice_to_management', 'review_pros', 'review_cons']
col_labels = {
    'advice_to_management': 'Advice to Management',
    'review_pros':          'Pros',
    'review_cons':          'Cons'
}
print('Null counts in text columns:')
for col in text_cols:
    print(f'  {col}: {df[col].isna().sum()} nulls')
df.head(3)

## Step 3 — Text Cleaning

We clean the text for keyword analysis — lowercasing, removing punctuation and numbers, stripping stopwords, and lemmatising words to their root form.

**Important:** the original columns stay untouched. Both sentiment models in Step 4 run on the raw text — cleaning removes words like `not` and `never` that both TextBlob and VADER need to detect negativity correctly.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

for col in text_cols:
    df[col + '_cleaned'] = df[col].apply(clean)

print('Cleaning done. Sample comparison:')
print()
print('Original Pros (row 0):'); print(df['review_pros'].iloc[0])
print()
print('Cleaned Pros (row 0):');  print(df['review_pros_cleaned'].iloc[0])

## Step 4 — Sentiment Scoring (TextBlob + VADER)

We run **two independent sentiment models** on each text column. Using a single model is a methodological weakness — if it misses signal, there is nothing to compare against. Running two lets us:
1. See where they agree (high confidence)
2. See where they disagree (ambiguous or factual complaint language)
3. Choose the more reliable model based on evidence from the validation step

**TextBlob** averages pre-assigned polarity values across all scored words. Simple, interpretable, but struggles with negation and factual complaints.

**VADER** is built for short-form social text. It handles capitalisation, punctuation emphasis, and negation (`not good` ≠ `good`) better than TextBlob. Standard VADER threshold: compound ≥ 0.05 = positive, ≤ -0.05 = negative.

Both models run on the **original, uncleaned text**.

In [ ]:
def get_textblob(text):
    if pd.isna(text) or str(text).strip() == '':
        return None, None
    score = round(TextBlob(str(text)).sentiment.polarity, 3)
    label = 'positive' if score > 0.1 else 'negative' if score < -0.1 else 'neutral'
    return score, label

def get_vader(text):
    if pd.isna(text) or str(text).strip() == '':
        return None, None
    compound = round(vader.polarity_scores(str(text))['compound'], 3)
    label    = 'positive' if compound >= 0.05 else 'negative' if compound <= -0.05 else 'neutral'
    return compound, label

for col in text_cols:
    df[[col + '_tb_score',  col + '_tb_label']]  = df[col].apply(lambda x: pd.Series(get_textblob(x)))
    df[[col + '_vdr_score', col + '_vdr_label']] = df[col].apply(lambda x: pd.Series(get_vader(x)))

print('Scoring done.')
print()
for col in text_cols:
    print(f'--- {col_labels[col]} ---')
    print('  TextBlob:', df[col + '_tb_label'].value_counts().to_dict())
    print('  VADER:   ', df[col + '_vdr_label'].value_counts().to_dict())
    print()

## Step 5 — Threshold Justification

Before trusting any labels, we look at where scores actually land. Label boundaries (±0.1 for TextBlob, ±0.05 for VADER) are choices, not facts. This chart shows whether those choices place labels sensibly given the shape of the data.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Score Distributions — TextBlob (top) vs VADER (bottom)',
             fontsize=13, fontweight='bold', y=1.02)

tb_colors  = ['#3498db', '#2ecc71', '#e74c3c']
vdr_colors = ['#1a6ea8', '#1aaa55', '#b03020']

for col_idx, (col, ct, cv) in enumerate(zip(text_cols, tb_colors, vdr_colors)):
    tb_scores  = df[col + '_tb_score'].dropna()
    vdr_scores = df[col + '_vdr_score'].dropna()

    ax = axes[0][col_idx]
    ax.hist(tb_scores, bins=25, color=ct, edgecolor='white', alpha=0.85)
    ax.axvline(x= 0.1, color='black', linestyle='--', linewidth=1.2, label='+0.1')
    ax.axvline(x=-0.1, color='black', linestyle=':',  linewidth=1.2, label='-0.1')
    ax.set_title(f'TextBlob — {col_labels[col]}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Polarity Score'); ax.set_ylabel('Reviews')
    ax.legend(fontsize=7); ax.spines[['top','right']].set_visible(False)
    pct = round(tb_scores.between(-0.1, 0.1).sum() / len(tb_scores) * 100, 1)
    ax.text(0.5, 0.92, f'{pct}% in neutral zone', transform=ax.transAxes,
            ha='center', fontsize=8, color='grey')

    ax2 = axes[1][col_idx]
    ax2.hist(vdr_scores, bins=25, color=cv, edgecolor='white', alpha=0.85)
    ax2.axvline(x= 0.05, color='black', linestyle='--', linewidth=1.2, label='+0.05')
    ax2.axvline(x=-0.05, color='black', linestyle=':',  linewidth=1.2, label='-0.05')
    ax2.set_title(f'VADER — {col_labels[col]}', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Compound Score'); ax2.set_ylabel('Reviews')
    ax2.legend(fontsize=7); ax2.spines[['top','right']].set_visible(False)
    pct2 = round(vdr_scores.between(-0.05, 0.05).sum() / len(vdr_scores) * 100, 1)
    ax2.text(0.5, 0.92, f'{pct2}% in neutral zone', transform=ax2.transAxes,
             ha='center', fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('chart0_thresholds.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Model Comparison: TextBlob vs VADER

We compare the two models on three measures:
1. **Score correlation** — do they agree on *how* positive or negative each review is?
2. **Label agreement** — do they assign the same label to the same review?
3. **Alignment with star ratings** — which model is more consistent with the reviewer's own score?

The model with higher star-rating alignment is the more reliable signal for this dataset.

In [ ]:
# Score correlation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('TextBlob vs VADER — Score Correlation per Column',
             fontsize=13, fontweight='bold', y=1.02)

for ax, col, color in zip(axes, text_cols, ['#3498db','#2ecc71','#e74c3c']):
    paired = df[[col+'_tb_score', col+'_vdr_score']].dropna()
    r, p   = stats.pearsonr(paired[col+'_tb_score'], paired[col+'_vdr_score'])
    ax.scatter(paired[col+'_tb_score'], paired[col+'_vdr_score'],
               alpha=0.4, s=18, color=color)
    ax.set_xlabel('TextBlob Score'); ax.set_ylabel('VADER Score')
    ax.set_title(f'{col_labels[col]}\nr = {r:.3f},  p = {p:.4f}',
                 fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'grey', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('chart_model_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Label agreement between models
print('=== LABEL AGREEMENT: TextBlob vs VADER ===')
print()
for col in text_cols:
    paired  = df[[col+'_tb_label', col+'_vdr_label']].dropna()
    agree   = (paired[col+'_tb_label'] == paired[col+'_vdr_label']).sum()
    total   = len(paired)
    pct     = round(agree / total * 100, 1)
    print(f'{col_labels[col]:25s}: {agree}/{total} agree ({pct}%)')
    disagree = paired[paired[col+'_tb_label'] != paired[col+'_vdr_label']]
    print(pd.crosstab(disagree[col+'_tb_label'], disagree[col+'_vdr_label'],
                      rownames=['TextBlob'], colnames=['VADER']))
    print()

In [ ]:
# Alignment with star ratings — the decisive comparison
df['rating_group'] = df['rating_overall'].apply(
    lambda x: 'high' if x >= 4 else ('low' if x <= 2 else 'mid')
)

print('=== ALIGNMENT WITH STAR RATINGS (Pros column) ===')
print('Mismatch = 4-5 star review with negative label OR 1-2 star review with positive label')
print()
sub = df[df['rating_group'].isin(['high','low'])].copy()
for model, label_col in [('TextBlob','review_pros_tb_label'), ('VADER','review_pros_vdr_label')]:
    mm = sub[
        ((sub['rating_group']=='high') & (sub[label_col]=='negative')) |
        ((sub['rating_group']=='low')  & (sub[label_col]=='positive'))
    ]
    pct = round((len(sub)-len(mm))/len(sub)*100, 1)
    print(f'  {model:10s}: {len(mm)} mismatches / {len(sub)} extreme-rated  →  Alignment: {pct}%')

print()
print('=== AVERAGE SCORE PER STAR RATING — BOTH MODELS, ALL COLUMNS ===')
for col in text_cols:
    print(f'\n{col_labels[col]}:')
    grp = df.groupby('rating_overall')[[col+'_tb_score', col+'_vdr_score']].mean().round(3)
    grp.columns = ['TextBlob','VADER']
    print(grp)

In [ ]:
# Comparison chart: avg score per star rating, both models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Average Sentiment Score per Star Rating — TextBlob vs VADER',
             fontsize=13, fontweight='bold', y=1.02)

for ax, col in zip(axes, text_cols):
    avg_tb  = df.groupby('rating_overall')[col+'_tb_score'].mean()
    avg_vdr = df.groupby('rating_overall')[col+'_vdr_score'].mean()
    ax.plot(avg_tb.index,  avg_tb.values,  marker='o', color='#3498db',
            linewidth=2, label='TextBlob')
    ax.plot(avg_vdr.index, avg_vdr.values, marker='s', color='#e74c3c',
            linewidth=2, linestyle='--', label='VADER')
    ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.8)
    ax.set_title(col_labels[col], fontsize=11, fontweight='bold')
    ax.set_xlabel('Star Rating'); ax.set_ylabel('Avg Score')
    ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('chart_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Both lines should trend upward — higher star ratings → higher sentiment scores.')
print('The model that tracks more cleanly is more reliable for this dataset.')

## Step 7 — Validation Against Star Ratings (Full — All 3 Columns)

The Pros column is the primary validation target but we check all three columns. The expected pattern:
- **Pros:** high-star → positive, low-star → less positive
- **Cons:** all reviews should lean negative, but low-star reviews should score *more* negative than high-star ones
- **Advice:** high-star workers typically give more constructive, less critical advice

We also run the manual spot check here — reading 20 reviews next to their model labels.

In [ ]:
# Cross-tab: star rating vs sentiment label — both models, all 3 columns
for col in text_cols:
    print(f'=== {col_labels[col].upper()} ===')
    print('TextBlob:')
    print(pd.crosstab(df['rating_overall'], df[col+'_tb_label']))
    print()
    print('VADER:')
    print(pd.crosstab(df['rating_overall'], df[col+'_vdr_label']))
    print()

In [ ]:
# Manual spot check — 20 reviews, both model labels side by side
# Read each row carefully and count how many labels look correct
spot = df[[
    'rating_overall',
    'review_pros',
    'review_pros_tb_label', 'review_pros_vdr_label',
    'review_cons',
    'review_cons_tb_label', 'review_cons_vdr_label'
]].sample(20, random_state=42).reset_index(drop=True)
spot.index += 1

pd.set_option('display.max_colwidth', 55)
spot

In [ ]:
# After reading the table above, fill in your counts
# A label 'looks correct' if the sentiment matches what a human reader would assign

tb_correct  = None   # <-- e.g. 15
vdr_correct = None   # <-- e.g. 17
sample_size = 20

if tb_correct is not None and vdr_correct is not None:
    print(f'TextBlob:  {tb_correct}/{sample_size} look correct ({round(tb_correct/sample_size*100,1)}%)')
    print(f'VADER:     {vdr_correct}/{sample_size} look correct ({round(vdr_correct/sample_size*100,1)}%)')
    better = 'VADER' if vdr_correct > tb_correct else 'TextBlob' if tb_correct > vdr_correct else 'both equal'
    print(f'Better on this sample: {better}')
    print()
    print('Use this alongside the star-rating alignment results to choose your primary model.')
else:
    print('Fill in tb_correct and vdr_correct after reading the spot check table.')

## Step 8 — Statistical Significance Tests

Observed differences in means are not meaningful without knowing whether they could occur by chance. We run three tests:

1. **Current vs Former — Overall Rating:** Welch's t-test (unequal variance assumed). Reports p-value and Cohen's d effect size.
2. **Sub-rating differences by employee status:** t-test on each dimension separately.
3. **Rating by tenure group:** Kruskal-Wallis (non-parametric — safer when group sizes may be unequal).

Significance threshold throughout: **p < 0.05**.

In [ ]:
# TEST 1: Current vs Former — Overall Rating
current = df[df['employee_status']=='Current Employee']['rating_overall'].dropna()
former  = df[df['employee_status']=='Former Employee']['rating_overall'].dropna()

t, p = ttest_ind(current, former, equal_var=False)
pooled_std = np.sqrt((current.std()**2 + former.std()**2) / 2)
d = round((current.mean()-former.mean()) / pooled_std, 3) if pooled_std > 0 else 0
effect = 'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'

print('=== TEST 1: Current vs Former Employee — Overall Rating ===')
print(f'Current:  n={len(current)},  mean={current.mean():.2f},  std={current.std():.2f}')
print(f'Former:   n={len(former)},   mean={former.mean():.2f},  std={former.std():.2f}')
print(f'Welch t:  t={t:.3f},  p={p:.4f}')
print(f"Cohen's d: {d}  ({effect} effect)")
print(f'Significant (p<0.05): {p < 0.05}')

In [ ]:
# TEST 2: Sub-rating differences by employee status
sub_ratings = [
    'rating_work_life', 'rating_culture_values',
    'rating_compensation_benefits', 'rating_senior_leadership',
    'rating_career_opportunities'
]

print('=== TEST 2: Sub-Rating Differences — Current vs Former ===')
print(f'{"Rating":<38} {"Current":>9} {"Former":>9} {"p-value":>9} {"Sig?":>6}')
print('-' * 75)

for col in sub_ratings:
    if col not in df.columns: continue
    c = df[df['employee_status']=='Current Employee'][col].dropna()
    f = df[df['employee_status']=='Former Employee'][col].dropna()
    if len(c) < 2 or len(f) < 2: continue
    t2, p2 = ttest_ind(c, f, equal_var=False)
    sig    = 'YES *' if p2 < 0.05 else 'no'
    print(f'{col:<38} {c.mean():>9.2f} {f.mean():>9.2f} {p2:>9.4f} {sig:>6}')

In [ ]:
# TEST 3: Rating by tenure group (Kruskal-Wallis)
if 'employee_length' in df.columns:
    print('=== TEST 3: Overall Rating by Tenure Group (Kruskal-Wallis) ===')
    groups = df.groupby('employee_length')['rating_overall'].apply(list).to_dict()
    for k, v in groups.items():
        print(f'  {str(k):<25}: n={len(v)}, mean={np.mean(v):.2f}')
    print()
    valid = [v for v in groups.values() if len(v) >= 3]
    if len(valid) >= 2:
        h, p_kw = kruskal(*valid)
        print(f'H = {h:.3f},  p = {p_kw:.4f}')
        print(f'Significant (p<0.05): {p_kw < 0.05}')
        if p_kw < 0.05:
            print('At least one tenure group rates Amazon significantly differently.')
        else:
            print('No significant difference across tenure groups.')
    else:
        print('Not enough groups with n≥3 for Kruskal-Wallis.')
else:
    print('employee_length column not found.')

In [ ]:
# Visualise: rating distribution by employee status
fig, ax = plt.subplots(figsize=(9, 5))

# Draw ALL lines first, THEN annotate — so ax.get_ylim() is stable
status_data = []
for status, color in [('Current Employee','#2471A3'),('Former Employee','#A93226')]:
    sub = df[df['employee_status']==status]['rating_overall'].dropna()
    counts = sub.value_counts().sort_index()
    ax.plot(counts.index, counts.values, marker='o', linewidth=2,
            color=color, label=f'{status} (n={len(sub)})')
    ax.axvline(x=sub.mean(), color=color, linestyle='--', linewidth=1, alpha=0.6)
    status_data.append((sub.mean(), color))

# ylim is now fully settled across both lines — safe to annotate
y_top = ax.get_ylim()[1]
for i, (m, color) in enumerate(status_data):
    y_pos = y_top * (0.88 if i == 0 else 0.72)
    ax.text(m + 0.05, y_pos, f'μ={m:.2f}', color=color, fontsize=9)

ax.set_title('Rating Distribution — Current vs Former Employees',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Star Rating (1–5)'); ax.set_ylabel('Reviews')
ax.legend(fontsize=10); ax.set_xticks([1,2,3,4,5])
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_rating_by_status.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Sentiment Visualizations

We show sentiment distributions for both models side by side. This makes it easy to see where the two models agree and where their label distributions diverge.

In [ ]:
color_map = {'positive':'#2ecc71','neutral':'#95a5a6','negative':'#e74c3c'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Sentiment Distribution — TextBlob (top) vs VADER (bottom)',
             fontsize=14, fontweight='bold', y=1.02)

for ci, col in enumerate(text_cols):
    for ri, (model, suffix) in enumerate([('TextBlob','_tb_label'),('VADER','_vdr_label')]):
        ax = axes[ri][ci]
        counts = df[col+suffix].value_counts().reindex(
            ['positive','neutral','negative'], fill_value=0)
        bars = ax.bar(counts.index, counts.values,
                      color=[color_map[l] for l in counts.index],
                      edgecolor='white', linewidth=1.5)
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
        ax.set_title(f'{model} — {col_labels[col]}', fontsize=10, fontweight='bold', pad=8)
        ax.set_ylabel('Reviews'); ax.set_ylim(0, counts.max()*1.25)
        ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('chart_sentiment_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Keyword Analysis

Each column is analysed separately — keywords from Pros stay separate from Cons. Mixing them would hide what employees say in each section.

Three levels: **unigrams** (what topics matter), **bigrams** (how workers frame them), **trigrams** (the most specific repeated phrases). An extended stopword list removes domain filler words that would dominate counts without adding meaning.

In [ ]:
extra_stops = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','have','has','had','do','does','did',
    'not','no','so','if','as','it','its','this','that','i','my','you','your',
    'we','our','they','their','he','she','very','just','get','got','can','will',
    'also','more','than','make','made','like','dont','lot','some','would',
    'could','really','much','many','one','even','take','go','well','im','ive'
])
all_stops = stop_words | extra_stops

def tokenize(text):
    return [w for w in re.findall(r"[a-z']{3,}", str(text).lower())
            if w not in all_stops]

def get_ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

keyword_data = {}
for col in text_cols:
    tokens_all = []
    for text in df[col+'_cleaned'].dropna():
        tokens_all.extend(tokenize(text))
    keyword_data[col] = {
        'uni':   Counter(tokens_all).most_common(20),
        'bi':    Counter(get_ngrams(tokens_all,2)).most_common(20),
        'tri':   Counter(get_ngrams(tokens_all,3)).most_common(15),
        'cloud': Counter(tokens_all).most_common(50)
    }

print('Keyword analysis complete.')

### Chart 1 — Top 20 Unigrams per Column

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Top 20 Unigrams by Section', fontsize=14, fontweight='bold', y=1.01)

for ax, col in zip(axes, text_cols):
    top    = keyword_data[col]['uni']
    words  = [w for w,c in top]; counts = [c for w,c in top]
    max_v  = max(counts) if counts else 1
    bar_cols = ['#1a1814' if c>=max_v*0.6 else '#c8922a' if c>=max_v*0.3 else '#9a9490'
                for c in counts]
    bars = ax.barh(words[::-1], counts[::-1], color=bar_cols[::-1], height=0.65)
    for bar, count in zip(bars, counts[::-1]):
        ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
                str(count), va='center', fontsize=8)
    ax.set_title(col_labels[col], fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency'); ax.spines[['top','right','left']].set_visible(False)

patches = [mpatches.Patch(color='#1a1814',label='High (top 60%)'),
           mpatches.Patch(color='#c8922a',label='Mid (30–60%)'),
           mpatches.Patch(color='#9a9490',label='Lower (<30%)')]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5,-0.05))
plt.tight_layout()
plt.savefig('chart1_unigrams.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 2 — Top 20 Bigrams per Column

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Top 20 Bigrams by Section', fontsize=14, fontweight='bold', y=1.01)
for ax, col in zip(axes, text_cols):
    top     = keyword_data[col]['bi']
    phrases = [p for p,c in top]; counts = [c for p,c in top]
    bars = ax.barh(phrases[::-1], counts[::-1], color='#2a5fa8', height=0.65)
    for bar, count in zip(bars, counts[::-1]):
        ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                str(count), va='center', fontsize=8)
    ax.set_title(col_labels[col], fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency'); ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout()
plt.savefig('chart2_bigrams.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 3 — Top 15 Trigrams per Column

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle('Top 15 Trigrams by Section', fontsize=14, fontweight='bold', y=1.01)
for ax, col in zip(axes, text_cols):
    top     = keyword_data[col]['tri']
    phrases = [p for p,c in top]; counts = [c for p,c in top]
    bars = ax.barh(phrases[::-1], counts[::-1], color='#6b3fa0', height=0.65)
    for bar, count in zip(bars, counts[::-1]):
        ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                str(count), va='center', fontsize=8)
    ax.set_title(col_labels[col], fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency'); ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout()
plt.savefig('chart3_trigrams.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 4 — Word Clouds per Column

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Word Cloud — Top 50 Keywords by Section',
             fontsize=14, fontweight='bold', y=1.02)
for ax, col, cmap in zip(axes, text_cols, ['Blues','Greens','Reds']):
    freq = dict(keyword_data[col]['cloud'])
    wc   = WordCloud(width=800, height=400, background_color='white',
                     colormap=cmap, max_words=50, collocations=False
                     ).generate_from_frequencies(freq)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(col_labels[col], fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('chart4_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Summary & Conclusions

This cell prints a structured summary of every finding in the pipeline. Run it after completing all steps above.

In [ ]:
print('=' * 65)
print('GLASSDOOR PIPELINE — SUMMARY')
print('=' * 65)
print()
print(f'Dataset:  {len(df)} reviews,  {df.shape[1]} columns')
print(f'Text columns: {text_cols}')
print(f'Null counts: {df[text_cols].isna().sum().to_dict()}')
print()
print('--- OVERALL RATINGS ---')
print(f'  Grand mean: {df["rating_overall"].mean():.2f}')
for s in df['employee_status'].dropna().unique():
    m = df[df['employee_status']==s]['rating_overall'].mean()
    n = df[df['employee_status']==s]['rating_overall'].count()
    print(f'  {s:<28}: mean={m:.2f},  n={n}')
print()
print('--- SENTIMENT DISTRIBUTIONS (VADER) ---')
for col in text_cols:
    dist = df[col+'_vdr_label'].value_counts(normalize=True).mul(100).round(1)
    print(f'  {col_labels[col]:25s}: {" | ".join([f"{k}: {v:.0f}%" for k,v in dist.items()])}')
print()
t3, p3 = ttest_ind(
    df[df['employee_status']=='Current Employee']['rating_overall'].dropna(),
    df[df['employee_status']=='Former Employee']['rating_overall'].dropna(),
    equal_var=False
)
print('--- STATISTICAL TESTS ---')
print(f'  Current vs Former overall:  p={p3:.4f}  →  {"SIGNIFICANT" if p3<0.05 else "not significant"}')
print()
print('--- MODEL COMPARISON (Pros column) ---')
paired = df[['review_pros_tb_score','review_pros_vdr_score']].dropna()
r4, p4 = stats.pearsonr(paired['review_pros_tb_score'], paired['review_pros_vdr_score'])
agree4 = (df['review_pros_tb_label']==df['review_pros_vdr_label']).mean()*100
print(f'  Score correlation:  r={r4:.3f},  p={p4:.4f}')
print(f'  Label agreement:    {agree4:.1f}%')
print()
print('--- LIMITATIONS ---')
print('  1. Both models are lexicon-based — sarcasm, irony, and factual')
print('     complaint language are systematically underscored.')
print('  2. Cons polarity near zero for both models — human labels more')
print('     reliable for that column.')
print('  3. Self-selected sample — not representative of all Amazon workers.')
print('  4. VADER was designed for social media; performance on longer')
print('     Glassdoor-style reviews may differ from benchmark scores.')

## Step 12 — Export Everything

In [ ]:
keep = [
    'review_id','rating_date','employee_status','employee_type',
    'employee_length','employee_location','employee_job_title',
    'rating_overall','rating_work_life','rating_culture_values',
    'rating_compensation_benefits','rating_senior_leadership',
    'rating_career_opportunities','rating_diversity_inclusion',
    'rating_group',
    'advice_to_management_cleaned','review_pros_cleaned','review_cons_cleaned',
    'advice_to_management_tb_score','advice_to_management_tb_label',
    'review_pros_tb_score','review_pros_tb_label',
    'review_cons_tb_score','review_cons_tb_label',
    'advice_to_management_vdr_score','advice_to_management_vdr_label',
    'review_pros_vdr_score','review_pros_vdr_label',
    'review_cons_vdr_score','review_cons_vdr_label',
]
available = [c for c in keep if c in df.columns]
df[available].to_csv('glassdoor_final_v2.csv', index=False)
print(f'Saved — {len(df)} rows, {len(available)} columns')

In [ ]:
from google.colab import files
files.download('glassdoor_final_v2.csv')
for chart in [
    'chart0_thresholds.png','chart_model_scatter.png','chart_model_comparison.png',
    'chart_rating_by_status.png','chart_sentiment_bars.png',
    'chart1_unigrams.png','chart2_bigrams.png','chart3_trigrams.png','chart4_wordclouds.png'
]:
    files.download(chart)
print('All files downloaded.')